In [0]:
import numpy as np
import pandas as pd
import mlflow

from mlflow.models import infer_signature

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import average_precision_score

from churn.config import GOLD_TABLE, ID, TARGET, RANDOM_SEED, MLFLOW_EXPERIMENT_NAME
from churn.validation import validar_gold
from churn.pipeline import construir_pipeline

usuario = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{usuario}/{MLFLOW_EXPERIMENT_NAME}")

In [0]:
pdf = spark.table(GOLD_TABLE).toPandas()
validar_gold(pdf)

X = pdf.drop(columns=[ID, TARGET])
y = pdf[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

In [0]:
model_LR = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
pipe = construir_pipeline(model_LR)

with mlflow.start_run(run_name="logreg_base") as run:

    mlflow.log_param("modelo", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("class_weight", "ninguno")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", RANDOM_SEED)
    mlflow.log_param("cv_n_splits", 5)

    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="average_precision")
    mlflow.log_metric("pr_auc_cv_media", scores.mean())
    mlflow.log_metric("pr_auc_cv_desviacion", scores.std())

    pipe.fit(X_train, y_train)
    ejemplo = X_train.head(3)
    firma = infer_signature(ejemplo, pipe.predict(ejemplo))

    mlflow.sklearn.log_model(
        sk_model=pipe,
        artifact_path="modelo",
        signature=firma,
        input_example=ejemplo,
    )

    y_proba = pipe.predict_proba(X_test)[:, 1]
    mlflow.log_metric("pr_auc_test", average_precision_score(y_test, y_proba))

In [0]:
run_id = mlflow.last_active_run().info.run_id

modelo_cargado = mlflow.pyfunc.load_model(f"runs:/{run_id}/modelo")

print(modelo_cargado.predict(X_test.head(3)))